In [3]:
import joblib
import sys

import pandas as pd
import numpy as np

sys.path.append("../scripts/twitter")
from searchTwitter import TwitterDataFrame
import mlp
from mlp import TweetDataset


from nltk.tokenize import TweetTokenizer
import spacy
import nltk
from nltk.stem import WordNetLemmatizer
import re
import emoji

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import matplotlib.pyplot as plt

import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

In [9]:
base_dir = "../data/"
cities = ["phoenix", "san_francisco", "portland", "seattle"]
locations = [("Maricopa", "Arizona", "phoenix"), ("San Francisco", "California", "san_francisco"), ("Multnomah", "Oregon", "portland"), ("King", "Washington", "seattle")]

In [10]:
aqi_df = mlp.make_aqi_df(base_dir, ['2018', '2020'], locations)
get_aqi = mlp.make_get_aqi(aqi_df)

c:\Users\Nick\Documents\P\bergin\wildfire\notebooks\../scripts/twitter\mlp.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aqi_temp["city"] = city
c:\Users\Nick\Documents\P\bergin\wildfire\notebooks\../scripts/twitter\mlp.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aqi_temp["city"] = city
c:\Users\Nick\Documents\P\bergin\wildfire\notebooks\../scripts/twitter\mlp.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_ind

In [11]:
tweet_df = mlp.make_tweet_df(base_dir, cities)
tweet_df.head()

c:\Users\Nick\Documents\P\bergin\wildfire\notebooks\../scripts/twitter\mlp.py:34: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs.append(pd.read_csv(base_dir + f))


,id,author_id,text,geo,created_at,lat,lon,city
0,1024082187115622400,1708905816,Time for Monday Night Raw! #RAW,{'place_id': '5c62ffb0f0f3479d'},2018-07-30 23:59:54,33.290260,-112.323914,phoenix
1,1024082144958636037,795028712408293376,@14bigjay88 @tate_slade_obsf they aren't gonna...,{'place_id': '0015d9147cee6907'},2018-07-30 23:59:44,33.384785,-112.357999,phoenix
2,1024082071222808577,275357706,@NicoleBarbaro Thanks for sharing! My younger ...,{'place_id': '7cb7440bcf83d464'},2018-07-30 23:59:26,33.319945,-111.979047,phoenix
3,1024082016969486337,20756054,@Telegraph I like all the people who are too b...,{'place_id': '5c62ffb0f0f3479d'},2018-07-30 23:59:14,33.290260,-112.323914,phoenix
4,1024081978864193536,601192710,"Adios, Arizona. Until next time 💔",{'place_id': '006b48995ede9bcc'},2018-07-30 23:59:04,33.204608,-111.842244,phoenix


In [6]:
spec_re = "[^A-Za-z0-9\@]+"
http_re = "https?:\S+|http?:\S"
at_re = "@\S+"

def process_tweet(tweet):
    tweet = str(tweet).lower()
    tweet = emoji.demojize(tweet)
    tweet = re.sub(spec_re, ' ', tweet)
    tweet = re.sub(http_re, 'HTTPURL', tweet)
    tweet = re.sub(at_re, '@USER', tweet)
    # Tokenize and stem?
    tweet = re.sub(r'\s+', ' ', tweet).strip()
    return tweet

tweet_df['text'] = tweet_df['text'].apply(process_tweet)

tweet_df.head()

,State Name,county Name,State Code,County Code,Date,AQI,Category,Defining Parameter,Defining Site,Number of Sites Reporting,date,text,Unnamed: 0,id,author_id,geo,created_at,lat,lon,city
0,Alabama,Baldwin,1.0,3.0,2018-01-02,32.0,Good,PM2.5,01-003-0010,1.0,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
1,Alabama,Baldwin,1.0,3.0,2018-01-05,34.0,Good,PM2.5,01-003-0010,1.0,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
2,Alabama,Baldwin,1.0,3.0,2018-01-08,15.0,Good,PM2.5,01-003-0010,1.0,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
3,Alabama,Baldwin,1.0,3.0,2018-01-11,19.0,Good,PM2.5,01-003-0010,1.0,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,
4,Alabama,Baldwin,1.0,3.0,2018-01-14,25.0,Good,PM2.5,01-003-0010,1.0,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,


In [7]:
vectorizer = CountVectorizer(stop_words='english', min_df=50, max_df=0.1)

In [8]:
train_df, test_df = train_test_split(tweet_df, test_size=0.2, random_state=42, stratify=tweet_df['date'])

get_vect = mlp.make_get_vect(train_df, vectorizer)
get_vect_test = mlp.make_get_vect(test_df, vectorizer)

ValueError: Input contains NaN

In [ ]:
X_train = TweetDataset(get_vect, get_aqi)
X_test = TweetDataset(get_vect_test, get_aqi, sample_rate=5)

train_loader = DataLoader(X_train, batch_size=1)
test_loader = DataLoader(X_test, batch_size=1)

In [ ]:
# Design mlp
class simple(nn.Module):

    def __init__(self, vocab):
        super().__init__()
        
        self.fc1 = nn.Linear(vocab, 100)
        self.fc2 = nn.Linear(100, 1)
        
    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        return self.fc2(h1)
    
    def loss_function(self, y, y_hat):
        MSE = (y - y_hat).pow(2).mean()
        return MSE


In [ ]:
vocab_size = len(vectorizer.vocabulary_)
model = simple(vocab_size)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-4)
epochs = 30
print_rate = 300

loss_results = mlp.train_model(model, train_loader, test_loader, optimizer, epochs, print_rate)

In [ ]:
model.eval()
y_vals = []
y_preds = []
for x, y in train_loader:
    x = x.to("cpu")
    y_vals.append(y)
    y_hat = model(x)
    y_preds.append(y_hat)
    
y_vals = torch.cat(y_vals).cpu().detach()
y_preds = torch.cat(y_preds).cpu().detach()
perfect = np.linspace(min(y_vals), max(y_vals), 100)
plt.plot(perfect, perfect, '--', color='red')
plt.xlabel("Predicted value")
plt.ylabel("Actual value")
plt.scatter(y_preds, y_vals)

In [ ]:
r2_score(y_preds, y_vals)
